## Part 1: Setup and Data Loading

In [17]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda:0
GPU: Tesla T4


In [ ]:
import pandas as pd
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jrobischon/wikipedia-movie-plots")

from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Hofstra/year3
%ls *.csv

#%cd /content/drive/MyDrive/TextMining/DataSets
#%ls *.csv

In [19]:
# Load movie data
movies_data = pd.read_csv('wiki_movie_plots_deduped.csv')
post1980 = movies_data[movies_data["Release Year"] > 1980]
print(f"Loaded {len(post1980)} movies from 1980+")

# Define queries from HW2
queries = [
    "scary movies to watch at night",
    "romantic comedy movies to watch for fun",
    "worst action movies of all time"
]

Loaded 19994 movies from 1980+


 Model 1: jina-embeddings-v2-base-en (ENCODER)
 - Context length of 2048/8192 tokens
 - Parameters: 137M
 - Base Model: BERT
 - Embedding Dimension 768
 - Rank: 157th
 - Zero-shot Coverage: 100%

 We chose this model b/c it meets the 500+ token requirements. Its 100% zero shot means it can perform well on embedding tasks without being specifically trained for the individual tasks.


Model 2: Qwen3-Embedding-0.6B
 - Context length of 1024/32768
 - Parameters: 595M
 - Base Model: Qwen3
 - Embedding Dimension 1024
 - Rank: 5
 - Zero-shot Coverage: 99%

I chose this model b/c its high model rank and high zero shot coverage, meaning it doesnt need to be trained specically well for tasks.

## Part 3: Load Models and Prepare Data

In [ ]:
# Load Model 1: Jina v3 (Encoder)
jina_model_name = "jinaai/jina-embeddings-v3"
jina_tokenizer = AutoTokenizer.from_pretrained(jina_model_name, trust_remote_code=True)
jina_model = SentenceTransformer(jina_model_name, device="cuda", trust_remote_code=True)


jina_model.max_seq_length = 2048

print(f"Loaded Jina v3")
print(f"Max sequence length: {jina_model.max_seq_length}")
print(f"Embedding dimension: {jina_model.get_sentence_embedding_dimension()}")

In [ ]:
# Load Model 2: Qwen3 (Decoder)
qwen_model_name = "Qwen/Qwen3-Embedding-0.6B"
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name, trust_remote_code=True)
qwen_model = SentenceTransformer(qwen_model_name, device="cuda", trust_remote_code=True)

qwen_model.max_seq_length = 1024

print(f"Loaded Qwen3 Embedding 0.6B")
print(f"Max sequence length: {qwen_model.max_seq_length}")
print(f"Embedding dimension: {qwen_model.get_sentence_embedding_dimension()}")

In [22]:
#extract plot and title data
all_plots = post1980['Plot'].tolist()
all_titles = post1980['Title'].tolist()
print(f"Total plots to embed: {len(all_plots)}")

#get first 10
first_10_plots = all_plots[:10]
first_10_titles = all_titles[:10]

Total plots to embed: 19994


## Part 4: Embed First 10 Plots

### Model 1: Jina v3 (Encoder)

How many movie plots were truncated by your embedding model in each model you selected

In [23]:
# Tokenize first 10 plots with Model 1
print("Tokenizing with Jina v3 (Encoder)\n")

tokens_jina = jina_model.tokenize(first_10_plots)
jina_token_counts = []
jina_truncated = []

max_len_jina = jina_model.max_seq_length

for i in range(len(first_10_plots)):
    token_ids = tokens_jina['input_ids'][i]
    actual_tokens = (token_ids != jina_tokenizer.pad_token_id).sum().item()
    jina_token_counts.append(actual_tokens)

    is_truncated = actual_tokens >= max_len_jina
    jina_truncated.append(is_truncated)

    status = "Truncated" if is_truncated else "OK"
    print(f"Plot {i+1}: {actual_tokens} tokens ({status})")
    print(f"  Title: {first_10_titles[i]}")

non_truncated_jina = sum(1 for t in jina_truncated if not t)
print(f"\nSummary: {non_truncated_jina}/10 plots embedded without truncation")
print(f"Truncation rate: {(10 - non_truncated_jina)/10 * 100:.1f}%")

Tokenizing with Jina v3 (Encoder)

Plot 1: 925 tokens (OK)
  Title: Absence of Malice
Plot 2: 531 tokens (OK)
  Title: All Night Long
Plot 3: 66 tokens (OK)
  Title: ...All the Marbles
Plot 4: 68 tokens (OK)
  Title: The Amateur
Plot 5: 997 tokens (OK)
  Title: American Pop
Plot 6: 1083 tokens (OK)
  Title: An American Werewolf in London
Plot 7: 127 tokens (OK)
  Title: Amy
Plot 8: 588 tokens (OK)
  Title: Arthur
Plot 9: 93 tokens (OK)
  Title: Back Roads
Plot 10: 1172 tokens (OK)
  Title: Blow Out

Summary: 10/10 plots embedded without truncation
Truncation rate: 0.0%


### Model 2: Qwen3 (Decoder)

In [24]:
# Tokenize first 10 plots with Model 2
print("Tokenizing with Qwen3 (Decoder)\n")

tokens_qwen = qwen_model.tokenize(first_10_plots)
qwen_token_counts = []
qwen_truncated = []

max_len_qwen = qwen_model.max_seq_length

for i in range(len(first_10_plots)):
    token_ids = tokens_qwen['input_ids'][i]
    actual_tokens = (token_ids != qwen_tokenizer.pad_token_id).sum().item()
    qwen_token_counts.append(actual_tokens)

    is_truncated = actual_tokens >= max_len_qwen
    qwen_truncated.append(is_truncated)

    status = "Truncated" if is_truncated else "OK"
    print(f"Plot {i+1}: {actual_tokens} tokens ({status})")
    print(f"  Title: {first_10_titles[i]}")

non_truncated_qwen = sum(1 for t in qwen_truncated if not t)
print(f"\nSummary: {non_truncated_qwen}/10 plots embedded without truncation")
print(f"Truncation rate: {(10 - non_truncated_qwen)/10 * 100:.1f}%")

Tokenizing with Qwen3 (Decoder)

Plot 1: 737 tokens (OK)
  Title: Absence of Malice
Plot 2: 465 tokens (OK)
  Title: All Night Long
Plot 3: 56 tokens (OK)
  Title: ...All the Marbles
Plot 4: 56 tokens (OK)
  Title: The Amateur
Plot 5: 838 tokens (OK)
  Title: American Pop
Plot 6: 946 tokens (OK)
  Title: An American Werewolf in London
Plot 7: 110 tokens (OK)
  Title: Amy
Plot 8: 511 tokens (OK)
  Title: Arthur
Plot 9: 82 tokens (OK)
  Title: Back Roads
Plot 10: 965 tokens (OK)
  Title: Blow Out

Summary: 10/10 plots embedded without truncation
Truncation rate: 0.0%


## Part 5: Word and Token Statistics for All Plots

In [25]:
# Calculate statistics for all plots
word_counts = []
for plot in all_plots:
    words = str(plot).split()
    word_counts.append(len(words))
#approx tokens per word (1.3 token to words)
token_counts_approx = [int(wc * 1.3) for wc in word_counts]

print(f"Total movies analyzed: {len(all_plots)}")
print(f"\nWord Count Statistics:")
print(f"  Maximum: {max(word_counts):,} words")
print(f"  Minimum: {min(word_counts):,} words")
print(f"  Average: {np.mean(word_counts):.1f} words")

print(f"\nApproximate Token Count Statistics:")
print(f"  Maximum: {max(token_counts_approx):,} tokens")
print(f"  Minimum: {min(token_counts_approx):,} tokens")
print(f"  Average: {np.mean(token_counts_approx):.1f} tokens")

Total movies analyzed: 19994

Word Count Statistics:
  Maximum: 6,752 words
  Minimum: 3 words
  Average: 425.0 words

Approximate Token Count Statistics:
  Maximum: 8,777 tokens
  Minimum: 3 tokens
  Average: 552.0 tokens


In [26]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
#empty out cache

## Part 6: Embed All Plots


In [27]:
'''# Embed all plots with Model 1 (Jina)
print("Embedding all plots with Jina v3...")
embeddings_jina = jina_model.encode(
    all_plots,
    batch_size=8,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=False
)

print(f"Jina embeddings shape: {embeddings_jina.shape}")
print(f"Memory usage: {embeddings_jina.nbytes / 1024**2:.1f} MB")'''
#commented out b/c saved to drive

'# Embed all plots with Model 1 (Jina)\nprint("Embedding all plots with Jina v3...")\nembeddings_jina = jina_model.encode(\n    all_plots,\n    batch_size=8,\n    show_progress_bar=True,\n    convert_to_numpy=True,\n    normalize_embeddings=False\n)\n\nprint(f"Jina embeddings shape: {embeddings_jina.shape}")\nprint(f"Memory usage: {embeddings_jina.nbytes / 1024**2:.1f} MB")'

In [28]:
'''# Embed all plots with Model 2 (Qwen)
print("Embedding all plots with Qwen3...")
embeddings_qwen = qwen_model.encode(
    all_plots,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=False
)

print(f"Qwen embeddings shape: {embeddings_qwen.shape}")
print(f"Memory usage: {embeddings_qwen.nbytes / 1024**2:.1f} MB")'''
#commented out b/c saved to drive

'# Embed all plots with Model 2 (Qwen)\nprint("Embedding all plots with Qwen3...")\nembeddings_qwen = qwen_model.encode(\n    all_plots,\n    batch_size=16,\n    show_progress_bar=True,\n    convert_to_numpy=True,\n    normalize_embeddings=False\n)\n\nprint(f"Qwen embeddings shape: {embeddings_qwen.shape}")\nprint(f"Memory usage: {embeddings_qwen.nbytes / 1024**2:.1f} MB")'

In [47]:
import numpy as np
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

# Load embeddings
embeddings_jina = np.load('/content/drive/MyDrive/Hofstra/year3/lab4/embeddings_jina.npy')
embeddings_qwen = np.load('/content/drive/MyDrive/Hofstra/year3/lab4/embeddings_qwen.npy')


print(f"Jina: {embeddings_jina.shape}")
print(f"Qwen: {embeddings_qwen.shape}")



Mounted at /content/drive
Jina: (19994, 1024)
Qwen: (19994, 1024)


 How many vectors you have stored in your faiss vector database

In [ ]:
!pip install -q faiss-cpu
import faiss


#normalize embeddings for cosine similarity
embeddings_jina_normalized = embeddings_jina.copy()
faiss.normalize_L2(embeddings_jina_normalized)

#get embedding dimension
embedding_dim_jina = embeddings_jina.shape[1]
print(f"Jina embedding dimension: {embedding_dim_jina}")

In [31]:
# Create Flat (Exact) index for Jina
index_jina_flat = faiss.IndexFlatIP(embedding_dim_jina)
index_jina_flat.add(embeddings_jina_normalized.astype('float32'))

print(f"Jina Flat Index created")
print(f"  Vectors stored: {index_jina_flat.ntotal:,}")

Jina Flat Index created
  Vectors stored: 19,994


In [32]:
# Create IVF (Approximate) index for Jina
nlist = 100  # number of clusters
quantizer_jina = faiss.IndexFlatIP(embedding_dim_jina)
index_jina_ivf = faiss.IndexIVFFlat(
    quantizer_jina,
    embedding_dim_jina,
    nlist,
    faiss.METRIC_INNER_PRODUCT
)

# Train the IVF index
print("Training Jina IVF index...")
index_jina_ivf.train(embeddings_jina_normalized.astype('float32'))
index_jina_ivf.add(embeddings_jina_normalized.astype('float32'))
index_jina_ivf.nprobe = 10  # number of clusters to search

print(f"Jina IVF Index created")
print(f"  Vectors stored: {index_jina_ivf.ntotal:,}")
print(f"  Clusters: {nlist}")
print(f"  nprobe: {index_jina_ivf.nprobe}")
print(f"  Index type: Approximate search using IVF")

Training Jina IVF index...
Jina IVF Index created
  Vectors stored: 19,994
  Clusters: 100
  nprobe: 10
  Index type: Approximate search using IVF


### Model 2: Qwen3 Indexes

In [33]:
# Normalize embeddings for cosine similarity
embeddings_qwen_normalized = embeddings_qwen.copy()
faiss.normalize_L2(embeddings_qwen_normalized)

# Get embedding dimension
embedding_dim_qwen = embeddings_qwen.shape[1]
print(f"Qwen embedding dimension: {embedding_dim_qwen}")

Qwen embedding dimension: 1024


In [34]:
# Create Flat (Exact) index for Qwen
index_qwen_flat = faiss.IndexFlatIP(embedding_dim_qwen)
index_qwen_flat.add(embeddings_qwen_normalized.astype('float32'))

print(f"Qwen Flat Index created")
print(f"  Vectors stored: {index_qwen_flat.ntotal:,}")
print(f"  Index type: Exact search using Inner Product")

Qwen Flat Index created
  Vectors stored: 19,994
  Index type: Exact search using Inner Product


In [50]:
# Create IVF (Approximate) index for Qwen
quantizer_qwen = faiss.IndexFlatIP(embedding_dim_qwen)
index_qwen_ivf = faiss.IndexIVFFlat(
    quantizer_qwen,
    embedding_dim_qwen,
    nlist,
    faiss.METRIC_INNER_PRODUCT
)

index_qwen_ivf.train(embeddings_qwen_normalized.astype('float32'))
index_qwen_ivf.add(embeddings_qwen_normalized.astype('float32'))
index_qwen_ivf.nprobe = 10

print(f"  Vectors stored: {index_qwen_ivf.ntotal:,}")
print(f"  Clusters: {nlist}")
print(f"  nprobe: {index_qwen_ivf.nprobe}")

  Vectors stored: 19,994
  Clusters: 100
  nprobe: 10


**Describe the main types of exact and ANN algorithms in the FAISS library and how you can select them.**

Approx vs Exact:
Exact computes distances to every vector, takes longer but gets more accurate results

Exact Indexes:
FlatL2: Euclidean distance
FlatIP: Inner product

ANN:
IVF: groups vecotrs into clusters, searches closest ones
HNSW: builds graph, has skipped linked list + NSW for quick loop up
PQ: Compresses vectors to save space
IVF + PQ: hybrid of clustering + compression

Selection:
Small: Flat
medium: IVF
Large: HNSW or hybrid

## Part 8: Query Evaluation

### Model 1: Jina v3 (Encoder) - Query Results

In [51]:
# Embed queries with encoder model (Jina)
query_embeddings_jina = jina_model.encode(
    queries,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print(f"Query embeddings shape: {query_embeddings_jina.shape}")

k = 7  # top 7 results
results_jina_exact = {}
results_jina_ivf = {}

# Exact search
print("ENCODER MODEL - EXACT SEARCH (Flat Index)")
print("--------------------------")

for i, query in enumerate(queries):
    print(f"\nQuery {i+1}: '{query}'")

    similarities, indices = index_jina_flat.search(
        query_embeddings_jina[i:i+1].astype('float32'), k
    )

    results_jina_exact[query] = []
    for rank, (sim, idx) in enumerate(zip(similarities[0], indices[0]), 1):
        title = post1980.iloc[idx]['Title']
        year = post1980.iloc[idx]['Release Year']
        plot = post1980.iloc[idx]['Plot']

        results_jina_exact[query].append({
            'rank': rank,
            'title': title,
            'year': year,
            'similarity': sim
        })

        print(f"  {rank}. {title} ({year}) - Similarity: {sim:.4f}")

# Approximate search
print("ENCODER MODEL - APPROXIMATE SEARCH (IVF Index)")
print("--------------------------")

for i, query in enumerate(queries):
    print(f"\nQuery {i+1}: '{query}'")

    similarities, indices = index_jina_ivf.search(
        query_embeddings_jina[i:i+1].astype('float32'), k
    )

    results_jina_ivf[query] = []
    for rank, (sim, idx) in enumerate(zip(similarities[0], indices[0]), 1):
        title = post1980.iloc[idx]['Title']
        year = post1980.iloc[idx]['Release Year']

        results_jina_ivf[query].append({
            'rank': rank,
            'title': title,
            'year': year,
            'similarity': sim
        })

        print(f"  {rank}. {title} ({year}) - Similarity: {sim:.4f}")

Query embeddings shape: (3, 1024)
ENCODER MODEL - EXACT SEARCH (Flat Index)
--------------------------

Query 1: 'scary movies to watch at night'
  1. They (2002) - Similarity: 0.6611
  2. Terror Tract (2000) - Similarity: 0.6534
  3. Terror in the Aisles (1984) - Similarity: 0.6362
  4. Cassandra (1986) - Similarity: 0.6277
  5. Deadly Dreams (1988) - Similarity: 0.6143
  6. Dark Tales of Japan (2004) - Similarity: 0.6043
  7. Tales from the Darkside: The Movie (1990) - Similarity: 0.6025

Query 2: 'romantic comedy movies to watch for fun'
  1. Lal Dupatta Malmal Ka (1989) - Similarity: 0.7176
  2. Manasina Maathu (2011) - Similarity: 0.6491
  3. Graduate (2011) - Similarity: 0.6451
  4. Premsutra (2013) - Similarity: 0.6293
  5. Kalgejje (2011) - Similarity: 0.6202
  6. Mudhal Kadhal Mazhai (2010) - Similarity: 0.6140
  7. Addicted to Love (1997) - Similarity: 0.6127

Query 3: 'worst action movies of all time'
  1. Laparwah (1981) - Similarity: 0.6052
  2. Why Don't You Play in Hell?

### Model 2: Qwen3 (Decoder) - Query Results

In [42]:
# Embed queries with Qwen model
query_embeddings_qwen = qwen_model.encode(
    queries,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print(f"Query embeddings shape: {query_embeddings_qwen.shape}")

results_qwen_exact = {}
results_qwen_ivf = {}

print("QWEN3 - EXACT SEARCH (Flat Index)")
print('----------------------------------')


for i, query in enumerate(queries):
    print(f"\nQuery {i+1}: '{query}'")

    similarities, indices = index_qwen_flat.search(
        query_embeddings_qwen[i:i+1].astype('float32'), k
    )

    results_qwen_exact[query] = []
    for rank, (sim, idx) in enumerate(zip(similarities[0], indices[0]), 1):
        title = post1980.iloc[idx]['Title']
        year = post1980.iloc[idx]['Release Year']

        results_qwen_exact[query].append({
            'rank': rank,
            'title': title,
            'year': year,
            'similarity': sim
        })

        print(f"  {rank}. {title} ({year}) - Similarity: {sim:.4f}")


print("QWEN3 - APPROXIMATE SEARCH (IVF Index)")
print('----------------------------------')

for i, query in enumerate(queries):
    print(f"\nQuery {i+1}: '{query}'")

    similarities, indices = index_qwen_ivf.search(
        query_embeddings_qwen[i:i+1].astype('float32'), k
    )

    results_qwen_ivf[query] = []
    for rank, (sim, idx) in enumerate(zip(similarities[0], indices[0]), 1):
        title = post1980.iloc[idx]['Title']
        year = post1980.iloc[idx]['Release Year']

        results_qwen_ivf[query].append({
            'rank': rank,
            'title': title,
            'year': year,
            'similarity': sim
        })

        print(f"  {rank}. {title} ({year}) - Similarity: {sim:.4f}")

Query embeddings shape: (3, 1024)
QWEN3 - EXACT SEARCH (Flat Index)
----------------------------------

Query 1: 'scary movies to watch at night'
  1. Haathkadi (1982) - Similarity: 0.5822
  2. Rasta (2003) - Similarity: 0.5508
  3. I Was a Teenage Zombie (1987) - Similarity: 0.5409
  4. Red Rain (2013) - Similarity: 0.5355
  5. Santrash (2003) - Similarity: 0.5220
  6. Abandoned Mine (2013) - Similarity: 0.5219
  7. Ghamandee (1981) - Similarity: 0.5218

Query 2: 'romantic comedy movies to watch for fun'
  1. Lal Dupatta Malmal Ka (1989) - Similarity: 0.6981
  2. Loop (1997) - Similarity: 0.6723
  3. Manasina Maathu (2011) - Similarity: 0.6630
  4. Thavarina Runa (2011) - Similarity: 0.6607
  5. Mudhal Kadhal Mazhai (2010) - Similarity: 0.6502
  6. A Saloon Wet with Beautiful Women (2002) - Similarity: 0.6441
  7. Min & Max (2016) - Similarity: 0.6338

Query 3: 'worst action movies of all time'
  1. Laparwah (1981) - Similarity: 0.5479
  2. Haathkadi (1982) - Similarity: 0.5462
  3. P

## Part 9: Precision Calculation



In [46]:
# Interactive Relevance Judgment
import numpy as np

relevance_judgments = {
    'encoder_flat': {},
    'encoder_ivf': {},
    'decoder_flat': {},
    'decoder_ivf': {}
}

query_labels = [
    "scary movies to watch at night",
    "romantic comedy movies to watch for fun",
    "worst action movies of all time"
]

methods = {
    'encoder_flat': ('Encoder', 'Flat', results_jina_exact),
    'encoder_ivf': ('Encoder', 'IVF', results_jina_ivf),
    'decoder_flat': ('Decoder', 'Flat', results_qwen_exact),
    'decoder_ivf': ('Decoder', 'IVF', results_qwen_ivf)
}

# Judge each result
for method_key, (model_name, search_type, results_dict) in methods.items():
    print(f"\n{model_name} - {search_type}")
    print("-" * 60)

    for query in query_labels:
        print(f"\nQuery: {query}")

        if query not in results_dict:
            print("No results found")
            relevance_judgments[method_key][query] = [0] * 7
            continue

        judgments = []
        results = results_dict[query]

        for result in results:
            idx = result.get('idx', result.get('doc_id'))
            print(f"\n{result['rank']}. {result['title']} ({result.get('year', 'N/A')})")

            # Get the plot from the dataframe
            if idx is not None:
                plot = post1980.iloc[idx]['Plot']
                print(f"Plot: {plot[:300]}...")

            while True:
                response = input("Relevant? (y/n): ").strip().lower()
                if response in ['y', 'n']:
                    judgments.append(1 if response == 'y' else 0)
                    break
                print("Enter y or n")

        relevance_judgments[method_key][query] = judgments
        precision = sum(judgments) / 7
        print(f"Precision: {sum(judgments)}/7 = {precision:.3f}")

# Calculate results
print("\n\nResults:")
print("=" * 60)

results_summary = {}

for method_key, (model_name, search_type, _) in methods.items():
    print(f"\n{model_name} - {search_type}:")
    method_precisions = []

    for i, query in enumerate(query_labels, 1):
        judgments = relevance_judgments[method_key][query]
        precision = sum(judgments) / 7
        method_precisions.append(precision)
        print(f"  Query {i}: {sum(judgments)}/7 = {precision:.3f}")

    avg_precision = np.mean(method_precisions)
    results_summary[method_key] = avg_precision
    print(f"  Average: {avg_precision:.3f}")

# Summary
encoder_avg = (results_summary['encoder_flat'] + results_summary['encoder_ivf']) / 2
decoder_avg = (results_summary['decoder_flat'] + results_summary['decoder_ivf']) / 2

print("\n\nFinal Averages:")
print(f"Encoder: {encoder_avg:.3f}")
print(f"Decoder: {decoder_avg:.3f}")

print("\nComparison to HW2/HW3:")
print(f"TF-IDF:   0.190")
print(f"BM25:     0.524")
print(f"Rocchio:  0.476")
print(f"Encoder:  {encoder_avg:.3f}")
print(f"Decoder:  {decoder_avg:.3f}")


Encoder - Flat
------------------------------------------------------------

Query: scary movies to watch at night

1. They (2002)
Relevant? (y/n): y

2. Terror Tract (2000)
Relevant? (y/n): y

3. Terror in the Aisles (1984)
Relevant? (y/n): y

4. Cassandra (1986)
Relevant? (y/n): y

5. Deadly Dreams (1988)
Relevant? (y/n): y

6. Dark Tales of Japan (2004)
Relevant? (y/n): y

7. Tales from the Darkside: The Movie (1990)
Relevant? (y/n): y
Precision: 7/7 = 1.000

Query: romantic comedy movies to watch for fun

1. Lal Dupatta Malmal Ka (1989)
Relevant? (y/n): y

2. Manasina Maathu (2011)
Relevant? (y/n): n

3. Graduate (2011)
Relevant? (y/n): n

4. Premsutra (2013)
Relevant? (y/n): n

5. Kalgejje (2011)
Relevant? (y/n): n

6. Mudhal Kadhal Mazhai (2010)
Relevant? (y/n): n

7. Addicted to Love (1997)
Relevant? (y/n): n
Precision: 1/7 = 0.143

Query: worst action movies of all time

1. Laparwah (1981)
Relevant? (y/n): n

2. Why Don't You Play in Hell? (2013)
Relevant? (y/n): y

3. Nightma

In [43]:
# Manually determined relevance for each query
# Replace these counts based on your actual results

# Query 1: "scary movies to watch at night"
jina_exact_q1_relevant = 7  # Fill in after manual review
jina_ivf_q1_relevant = 7
qwen_exact_q1_relevant = 6
qwen_ivf_q1_relevant = 6

# Query 2: "romantic comedy movies to watch for fun"
jina_exact_q2_relevant = 5
jina_ivf_q2_relevant = 5
qwen_exact_q2_relevant = 6
qwen_ivf_q2_relevant = 6

# Query 3: "worst action movies of all time"
jina_exact_q3_relevant = 4
jina_ivf_q3_relevant = 5
qwen_exact_q3_relevant = 4
qwen_ivf_q3_relevant = 4

# Calculate precisions
precision_jina_exact = [
    jina_exact_q1_relevant/7,
    jina_exact_q2_relevant/7,
    jina_exact_q3_relevant/7
]

precision_jina_ivf = [
    jina_ivf_q1_relevant/7,
    jina_ivf_q2_relevant/7,
    jina_ivf_q3_relevant/7
]

precision_qwen_exact = [
    qwen_exact_q1_relevant/7,
    qwen_exact_q2_relevant/7,
    qwen_exact_q3_relevant/7
]

precision_qwen_ivf = [
    qwen_ivf_q1_relevant/7,
    qwen_ivf_q2_relevant/7,
    qwen_ivf_q3_relevant/7
]

# Print results
print("PRECISION @ 7 RESULTS\n")
print("="*80)
print("Model 1: Jina v3 (Encoder)")
print("="*80)
for i, query in enumerate(queries):
    print(f"\nQuery {i+1}: '{query}'")
    print(f"  Exact (Flat):       {precision_jina_exact[i]:.3f}")
    print(f"  Approximate (IVF):  {precision_jina_ivf[i]:.3f}")

print(f"\nAverage Precision:")
print(f"  Exact (Flat):       {np.mean(precision_jina_exact):.3f}")
print(f"  Approximate (IVF):  {np.mean(precision_jina_ivf):.3f}")

print("\n" + "="*80)
print("Model 2: Qwen3 (Decoder)")
print("="*80)
for i, query in enumerate(queries):
    print(f"\nQuery {i+1}: '{query}'")
    print(f"  Exact (Flat):       {precision_qwen_exact[i]:.3f}")
    print(f"  Approximate (IVF):  {precision_qwen_ivf[i]:.3f}")

print(f"\nAverage Precision:")
print(f"  Exact (Flat):       {np.mean(precision_qwen_exact):.3f}")
print(f"  Approximate (IVF):  {np.mean(precision_qwen_ivf):.3f}")

PRECISION @ 7 RESULTS

Model 1: Jina v3 (Encoder)

Query 1: 'scary movies to watch at night'
  Exact (Flat):       1.000
  Approximate (IVF):  1.000

Query 2: 'romantic comedy movies to watch for fun'
  Exact (Flat):       0.714
  Approximate (IVF):  0.714

Query 3: 'worst action movies of all time'
  Exact (Flat):       0.571
  Approximate (IVF):  0.714

Average Precision:
  Exact (Flat):       0.762
  Approximate (IVF):  0.810

Model 2: Qwen3 (Decoder)

Query 1: 'scary movies to watch at night'
  Exact (Flat):       0.857
  Approximate (IVF):  0.857

Query 2: 'romantic comedy movies to watch for fun'
  Exact (Flat):       0.857
  Approximate (IVF):  0.857

Query 3: 'worst action movies of all time'
  Exact (Flat):       0.571
  Approximate (IVF):  0.571

Average Precision:
  Exact (Flat):       0.762
  Approximate (IVF):  0.762


## Part 10: Comparison and Analysis

### Results Summary

- TF-IDF: 0.190
- BM25: 0.524
- Rocchio: 0.476


- Encoder (Jina): 0.690 average
  - Flat: 0.619 | IVF: 0.762
- Decoder (Qwen3): 0.595 average
  - Flat: 0.619 | IVF: 0.571

### Query-Level Results

**Query 1 (horror movies):** Embeddings excelled with 0.857-1.000 precision vs. 0.429 for traditional methods.

**Query 2 (romantic comedy):** Decoder performed better (0.429-0.571) than encoder (0.143-0.286). Both beat TF-IDF's 0.000.

**Query 3 (worst action):** Highly variable. Encoder IVF achieved 1.000, decoder flat only 0.429.

### Key Findings

Encoder outperformed the decoder based on precision. Encoder beating decoder is reasonable bc they are meant for understanding meaning in both directions while decoders are mainly for text generation. Encoder seemed to have out performed the other model types. Some reason alot of indian movies were selected.

### Conclusion

Sentence embeddings provide better understandings then normal keyword based IR methods. They resulted with higher precision and overall embeddings are a strong upgrade for semantic search.